In [ ]:
# In this script, the illustrations for linear data with a multimodal residual are created
import numpy as np
import torch
from scipy.stats import norm, bernoulli
from lidglm import ExtendedGLM_Utils
from lidglm.extended_glm import TransformerUtils
import statsmodels.api as sm
import matplotlib.pyplot as plt
import pandas as pd
plt.rcParams['text.usetex'] = True

np.random.seed(42)
torch.manual_seed(42)

In [ ]:
#We generate a 1-dimensional covariate and a target with a linearly dependent mean + 2 normal distributed errors

num_samples = 10**5
covariate=np.linspace(-1,1,num_samples)
#x= np.ones(num_samples)

loc_1 = -0.5
scale_1 = 0.3
loc_2 = 0.5
scale_2 = 0.3

def function(x):
    return 4*x + 3

def add_noise(target, loc_1, scale_1, loc_2, scale_2):
    target_length = len(target) if isinstance(target, np.ndarray) else 1
    which_mode = bernoulli.rvs(0.5, size=target_length)
    noise = np.where(which_mode, 
                     norm.rvs(size=target_length, loc=loc_1, scale=scale_1), 
                     norm.rvs(size=target_length, loc=loc_2, scale=scale_2))
    return target + noise

target = add_noise(function(covariate), loc_1, scale_1, loc_2, scale_2)


In [ ]:
#fit data with a traditional GLM

num_params = 1
traditional_glm = sm.GLM(endog=target, exog=sm.add_constant(covariate, prepend=True), family=sm.families.Gaussian() )
traditional_results=traditional_glm.fit(maxiter=10**4, scale = "dev")

In [ ]:
# Now we construct a LiD-GLM:
#first we define the i-ResNet for the transformation of the endogeneous variable (the target)
net_transform_endog = TransformerUtils().initialize_transformer(num_params=1, lipschitz_const=3.99, width=16, depth=3, domain_codomain=1,  activation_string = "swish", group_size = 8)


#We convert the base GLM to a LiD-GLM and append the i-ResNet
util = ExtendedGLM_Utils()
extended_glm, target_tensor, covariate_tensor = util.convert_sm_glm(glm=traditional_results, net_transform_endog=net_transform_endog, bias = "first")

In [ ]:
#Fit our model:
extended_glm.to("cpu")
covariate_tensor = covariate_tensor.to("cpu")
target_tensor = target_tensor.to("cpu")

loss_hist = util.train_model(extended_glm, exog=covariate_tensor, endog=target_tensor, epochs=400, lr=4*0.001, loss_crit = "log_like", early_stopping=True, part_to_train="only_y_transform", patience=30)[0]
plt.plot(loss_hist)
extended_glm.to("cpu")
extended_glm.set_dispersion(exog=covariate_tensor, endog=target_tensor)

In [ ]:
# Plot GLM vs LiD-GLM distribution vs true distribution to show how LiD-GLM is able to learn a multimodal target distribution (Figure 3)
extended_glm.to("cpu")
extended_glm.dispersion = extended_glm.dispersion.to("cpu")
fig, ax = plt.subplots(figsize=(6,5))

ax.hist2d(covariate, target, bins=500, cmap='Grays') # ToDo: add label
#ax.scatter(covariate, target, s=1, alpha=0.009, color='gray')
# we insert a dummy entry to get the legend right
ax.scatter([], [], color='gray', marker='o', s=12, label="Data")

ax.plot(covariate, function(covariate), color = "black", label = "True mean")
#ax.plot(covariate,extended_glm.predict_endog(covariate_tensor).detach().numpy(),c ="blue", label="predicted mean")

intervals = [-0.75, -0.25, 0.25, 0.75]
true_densities = []
glm_densities = []
lid_glm_densities = []
for i in intervals:
    interval_exog = i* torch.ones_like(covariate_tensor, device = "cpu")

    lid_glm_density = extended_glm.get_distribution(exog=interval_exog, dispersion=extended_glm.dispersion)
    untransf_endog_sample = lid_glm_density.sample().reshape(-1,1)
    transf_endog_sample = (extended_glm.net_transform_endog(untransf_endog_sample-extended_glm.predict(interval_exog).reshape(-1,1))[0]+extended_glm.predict(interval_exog).reshape(-1,1)).reshape(-1).detach().cpu().numpy()
    constant_covariates = np.ones(num_samples) * i
    single_density = add_noise(function(constant_covariates), loc_1, scale_1, loc_2, scale_2)
#
    true_densities.append(single_density)
    lid_glm_densities.append(transf_endog_sample)

    # we also get the glm distribution:
    traditional_distr = traditional_results.get_distribution(sm.add_constant(interval_exog.detach().cpu().numpy(), has_constant="add", prepend=True))
    glm_densities.append(traditional_distr.rvs())

def add_label_violinplot(violinplot, label):
    color = violinplot['bodies'][0].get_edgecolor()
    # append to labels
    violinplot['bodies'][0].set_label(label)


true_densities = pd.DataFrame(true_densities).T
lid_glm_densities = pd.DataFrame(lid_glm_densities).T
glm_densities = pd.DataFrame(glm_densities).T
true_densities_plots = ax.violinplot(true_densities, positions=intervals, showmeans=False, showextrema=False, points=5000)
add_label_violinplot(true_densities_plots, "True Density")
lid_glm_density_plots = ax.violinplot(lid_glm_densities, positions=intervals, side = 'high', showmeans=True, showextrema=False, points=5000)
add_label_violinplot(lid_glm_density_plots, "LiD-GLM Density")
glm_density_plots = ax.violinplot(glm_densities, positions=intervals, points=5000, side = 'low', showmeans=True, showextrema=False)
add_label_violinplot(glm_density_plots, "GLM Density")
for pc in lid_glm_density_plots['bodies']:
    pc.set_facecolor('blue')
    pc.set_alpha(0.3)
    pc.set_edgecolor('blue')
lid_glm_density_plots['cmeans'].set_edgecolor('blue')

for pc in true_densities_plots['bodies']:
    pc.set_edgecolor("black")
    pc.set_alpha(0.6)
    pc.set_linewidth(1.5)
    #set face transparent
    pc.set_facecolor("none")
    pc.set_linestyle("--")

for pc in glm_density_plots["bodies"]:
    pc.set_facecolor('red')
    pc.set_edgecolor('red')
    pc.set_alpha(0.3)
    # make the face dashed:
    pc.set_hatch('///')
    
glm_density_plots['cmeans'].set_edgecolor('red')
    



ax.set_xlabel("Covariate $x$", fontsize =15)
ax.set_ylabel("Target $y$", fontsize =15)
ax.set_xlim(-1.05, 1.05)
fig.legend(loc = (0.15, 0.7), fontsize = 13)
fig.tight_layout()
fig.savefig("multimodal_resid_comparis_glm_lidglm.png", bbox_inches='tight', dpi = 1200)


In [ ]:
# We now train a LiD-Glm on the multimodal data for 3 different Lipschitz bounds to show the ability to control the strength of the distributional correction below (Fig. 4)
util = ExtendedGLM_Utils()
Lipschitz_constants = [1.45, 1.99, 3.99]
extended_glms = []

for i, constant in enumerate(Lipschitz_constants):
    # train an extended_glm:
    net_transform_endog = TransformerUtils().initialize_transformer(num_params=1, lipschitz_const=constant, width=16, depth=3, domain_codomain=2,  activation_string = "swish")
    extended_glm, target_tensor, covariate_tensor = util.convert_sm_glm(glm=traditional_results, net_transform_endog=net_transform_endog, bias = "first")
    extended_glm.to("cpu")
    covariate_tensor = covariate_tensor.to("cpu")
    target_tensor = target_tensor.to("cpu")
    loss_hist = util.train_model(extended_glm, exog=covariate_tensor, endog=target_tensor, epochs=400, lr=4*0.001, loss_crit = "log_like", early_stopping=True, part_to_train="only_y_transform", patience=30)[0]
    extended_glm.eval()
    extended_glms.append(extended_glm)

In [ ]:
fig, axs = plt.subplots(1, 3 , figsize=(11,3.1), sharey=True)
n_plot_points = 10**6
x= torch.ones(n_plot_points,1)
y_plot = torch.linspace(5,9,n_plot_points)
true_mean = function(x).reshape(-1)[0]
density_plot = (norm.pdf(y_plot.detach().numpy(), loc = loc_1+true_mean, scale= scale_1)*0.5 
                + norm.pdf(y_plot.detach().numpy(), loc = loc_2+true_mean, scale=scale_2)*0.5)

covariate_tensor = covariate_tensor.to("cpu")
target_tensor = target_tensor.to("cpu")
for i, extended_glm in enumerate(extended_glms):
    ax = axs[i]
    extended_glm.to("cpu")
    temporary_mu = extended_glm.predict(covariate_tensor).reshape(-1)
    temporary_mu_transf = extended_glm.predict_endog(covariate_tensor).reshape(-1)
    temporary_backtransf_residual = extended_glm.net_transform_endog.inverse((target_tensor-temporary_mu_transf).reshape(-1,1))[0].reshape(-1)



    density = torch.exp(extended_glm.log_like_obs(endog= y_plot, exog = x, dispersion = extended_glm.glm.family.dispersion(
                                endog=temporary_backtransf_residual + temporary_mu,
                                mu=temporary_mu)).detach())
    untransf_density = torch.exp(extended_glm.log_like_untransformed(endog= y_plot, exog = x, dispersion = extended_glm.glm.family.dispersion(
                                endog=target_tensor,
                                mu=temporary_mu)).detach())

    ax.plot(y_plot.detach().numpy(), density.numpy(), c="red", linewidth = 1.5, label = "LiD-GLM")
    ax.plot(y_plot.detach().numpy(), traditional_results.get_distribution(sm.add_constant(x.detach().numpy(), prepend =False, has_constant='add')).pdf(y_plot.detach().numpy()), 
            linestyle = ":", linewidth = 2, label = "GLM")
    ax.plot(y_plot.detach().numpy(), density_plot, c="black", label = "True Density", alpha = 0.6, linestyle = "--", linewidth = 2)
    
    ax.set_xlabel("$y$", fontsize = 15)
    ax.set_ylim(0,0.8)
    ax.set_xlim(5,9)
    letters = ["(a)", "(b)", "(c)"]
    ax.set_title(letters[i] + "  " + "$L^b_d=$ " + str(np.round(Lipschitz_constants[i]-1, 2)), fontsize = 13)
axs[0].set_ylabel("$f_Y(y)$", fontsize = 15)
axs[2].legend(fontsize = 12, loc = (1.01,0.64))
fig.tight_layout()
fig.savefig("multimodal_ex_lipschitz_compar.pdf", bbox_inches='tight')